In [25]:
from kilosort_pipeline.utils import setup, parse_openephys_folders
from kilosort_pipeline.sync import match_chirp_edges
import spikeinterface.extractors as se
import spikeinterface as si

from pathlib import Path
from collections import defaultdict
from loguru import logger
import numpy as np
import pynapple as nap
from scipy.interpolate import make_interp_spline

def log_ts(timestamps, name):
    start_ts = timestamps[0]
    end_ts = timestamps[-1]
    logger.info(f"{name}: {start_ts:.4f} ... {end_ts:.4f} s ({timestamps.size} samples)")

def load_events(events_path, cont_path):
    event_ts  = np.load(events_path, mmap_mode='r')
    cont_ts   = np.load(cont_path, mmap_mode='r')
    states    = np.load(events_path.replace('timestamps.npy', 'states.npy'), mmap_mode='r')
    return event_ts, cont_ts, states

def get_kilosort_spikes(output_path, probe_filter=None):
    spike_times_dict = {}

    kilosort_files = list(Path(output_path).glob('*/kilosort/spike_times.npy'))
    if not kilosort_files:
        logger.error("No Kilosort output found. Run Kilosort first.")
        raise FileNotFoundError(f"No spike_times.npy files in {output_path}")
    
    for spike_file in kilosort_files:
        probe_name = spike_file.parent.parent.name

        # Filter probes if requested
        if probe_filter and probe_name not in probe_filter:
            continue

        spike_times_dict[probe_name] = np.load(spike_file, mmap_mode='r')
        logger.info(f"Loaded {len(spike_times_dict[probe_name])} spikes from {probe_name}")

    return spike_times_dict

In [2]:
conf_path = Path('R:\\Basic_Sciences\\Phys\\SenzaiLab\\kilosort_output\\configs\\config.yaml')
p = setup(conf_path)
ps = parse_openephys_folders(p['recording_paths'], p['probe_filter'])

2025-11-03 13:39:31 | SUCCESS  | Loaded configuration from: R:\Basic_Sciences\Phys\SenzaiLab\kilosort_output\configs\config.yaml
2025-11-03 13:39:31 | SUCCESS  | Logger configured at E:\kilosort_output\AA001_Day2\AA001_Day2_20251103_133931.log
2025-11-03 13:39:31 | INFO     | Pipeline configuration:
2025-11-03 13:39:31 | INFO     | Session: AA001_Day2
2025-11-03 13:39:31 | INFO     | Recording sessions: 4
2025-11-03 13:39:31 | INFO     | Local output: E:\kilosort_output\AA001_Day2
2025-11-03 13:39:31 | INFO     | Remote output: \\fsmresfiles.fsm.northwestern.edu\FSMResfiles\Basic_Sciences\Phys\SenzaiLab\kilosort_output\AA001_Day2
2025-11-03 13:39:31 | INFO     | Parsing OpenEphys folders
2025-11-03 13:40:22 | SUCCESS  | Parsed 5 stream(s) across 4 session(s)


In [3]:
ks_spikes = get_kilosort_spikes(output_path=p['local_output'], probe_filter=['ProbeA', 'ProbeB'])

2025-11-03 13:40:22 | INFO     | Loaded 110944227 spikes from ProbeA
2025-11-03 13:40:22 | INFO     | Loaded 164597833 spikes from ProbeB


In [72]:
adc = se.read_openephys(r"R:\Basic_Sciences\Phys\SenzaiLab\Ayo\AA002\Day2\OpenField\AA002_2025-10-31_13-45-48_2Probe_ADn_RecOpenField", stream_id='Record Node 124#OneBox-122.ProbeASYNC')
adc.get_times()

array([4.40916817e-01, 4.40950150e-01, 4.40983483e-01, ...,
       1.15200468e+04, 1.15200468e+04, 1.15200469e+04], shape=(345588179,))

In [60]:
se.get_neo_streams('openephysbinary', r"R:\Basic_Sciences\Phys\SenzaiLab\Ayo\AA002\Day2\OpenField\AA002_2025-10-31_13-45-48_2Probe_ADn_RecOpenField")

(['Record Node 124#OneBox-122.OneBox-ADC',
  'Record Node 124#OneBox-122.ProbeA',
  'Record Node 124#OneBox-122.ProbeB',
  'Record Node 124#OneBox-122.ProbeASYNC',
  'Record Node 124#OneBox-122.ProbeBSYNC'],
 ['0',
  '1',
  '2',
  'Record Node 124#OneBox-122.ProbeASYNC',
  'Record Node 124#OneBox-122.ProbeBSYNC'])

In [53]:
path = r"R:\Basic_Sciences\Phys\SenzaiLab\Ayo\AA002\Day2\OpenField\AA002_2025-10-31_13-45-48_2Probe_ADn_RecOpenField\Record Node 124\experiment1\recording1\continuous\OneBox-122.OneBox-ADC\timestamps.npy"
path2 = r"R:\Basic_Sciences\Phys\SenzaiLab\Ayo\AA002\Day2\OpenField\AA002_2025-10-31_13-45-48_2Probe_ADn_RecOpenField\Record Node 124\experiment1\recording1\continuous\OneBox-122.OneBox-ADC\continuous.dat"
ts = np.load(path, mmap_mode='r')
ts

memmap([4.00796236e-01, 4.00829235e-01, 4.00862233e-01, ...,
        1.15199991e+04, 1.15199991e+04, 1.15199992e+04],
       shape=(349078682,))

In [64]:
adc.get_time_info()

{'sampling_frequency': 30300.5,
 't_start': np.float64(0.4409168165541823),
 'time_vector': None}

In [203]:
class Timestamps:
    def __init__(self, name, fs, t_start=0.0):
        self.name = name
        self.fs = fs
        self.dt = 1 / fs
        
        self.global_timestamps = []
        self.intervals = []
        self.t_offset = t_start
        self.starting_states = []

    def update(self, local_ts, t_end, starting_state=None):
        """ Update global timestamps with a new segment."""
        # Update global timestamps
        self.global_timestamps.append(local_ts + self.t_offset + self.dt)

        # Update intervals
        self.intervals.append((self.t_offset, self.t_offset + t_end))
        
        # Update offset for next segment
        self.t_offset += t_end

        if starting_state:
            self.starting_states.append(starting_state)
        
    def __repr__(self):
        return f"Timestamps(name='{self.name}', @ {self.fs:.1f} Hz)"

In [ ]:
ADC = Timestamps(name='OneBox-ADC', fs=30300.5, t_start=0.0)

adc_event_paths = ps["timestamps"]["OneBox-ADC"]['event']
adc_cont_paths = ps["timestamps"]["OneBox-ADC"]['cont']

output_path = protocol['local_output']/'OneBox-ADC'/'timestamps.npy'
t_last = 0.0


for idx, (event_path, cont_path) in enumerate(zip(adc_event_paths, adc_cont_paths)):
    event_ts, cont_ts, states = load_events(event_path, cont_path)

    # Subtract the offset of continuous recording
    ADC.update(event_ts - cont_ts[0], cont_ts[-1] - cont_ts[0], starting_state=states[0])

    ########## LOGGING #################################
    log_ts(event_ts, "ADC event")
    log_ts(cont_ts, "ADC cont")
    logger.info(f"ADC interval: {ADC.intervals[-1][0]:.4f} ... {ADC.intervals[-1][1]:.4f} s")
    log_ts(ADC.global_timestamps[-1], "ADC global segment")
    log_ts(np.concatenate(ADC.global_timestamps), "ADC global")
    logger.info("-"*60)
    ####################################################
    
    # Save timestamps
    cont_ts_shifted = cont_ts - cont_ts[0] + t_last
    if idx == 0:
        # Create new file
        np.save(output_path, cont_ts_shifted)
    else:
        # Append to existing file
        existing = np.load(output_path, mmap_mode='r')
        np.save(output_path, np.concatenate((existing, cont_ts_shifted)))
        del existing


2025-11-03 18:18:56 | INFO     | ADC event: 12.0830 ... 3645.1092 s (7267 samples)
2025-11-03 18:18:56 | INFO     | ADC cont: 11.6334 ... 3645.4310 s (110114290 samples)
2025-11-03 18:18:56 | INFO     | ADC interval: 0.0000 ... 3633.7977 s
2025-11-03 18:18:56 | INFO     | ADC global segment: 0.4497 ... 3633.4759 s (7267 samples)
2025-11-03 18:18:56 | INFO     | ADC global: 0.4497 ... 3633.4759 s (7267 samples)
2025-11-03 18:18:56 | INFO     | ------------------------------------------------------------
2025-11-03 18:18:56 | INFO     | ADC event: 123.0811 ... 2266.3990 s (11788 samples)
2025-11-03 18:18:56 | INFO     | ADC cont: 123.0448 ... 2266.4321 s (64945708 samples)
2025-11-03 18:18:56 | INFO     | ADC interval: 3633.7977 ... 5777.1850 s
2025-11-03 18:18:56 | INFO     | ADC global segment: 3633.8339 ... 5777.1519 s (11788 samples)
2025-11-03 18:18:56 | INFO     | ADC global: 0.4497 ... 5777.1519 s (19055 samples)
2025-11-03 18:18:56 | INFO     | -----------------------------------

In [243]:
probe_filter = ['ProbeA', 'ProbeB']
probe_timestamps = {k:d for k,d in ps["timestamps"].items() if k != "OneBox-ADC" and k in probe_filter}

masks = {}
for probe, paths in probe_timestamps.items():
    PRB = Timestamps(name=probe, fs=30000.0, t_start=0.0)
    adc_global_timestamps = []
    logger.info(f"Processing probe: {probe}")

    logger.info("Extracting kilosort spikes")
    kilosort_spikes = ks_spikes[probe] / PRB.fs
    log_ts(kilosort_spikes, f"{probe} spikes")
    total_spikes_left = kilosort_spikes.size
    logger.info('='*60)

    save_dir = Path(p['local_output'] / probe)
    save_dir.mkdir(parents=True, exist_ok=True)
    logger.info(f"Saving to: {save_dir}")

    masks           = []
    synced_spikes   = []

    for idx, (ev_path, cont_path) in enumerate(zip(paths['event'], paths['cont'])):
        event_ts, cont_ts, states = load_events(ev_path, cont_path)

        # Handle state mismatches
        if states[0] != ADC.starting_states[idx]:
            logger.warning(f"State mismatch between {probe} and ADC")
            logger.info("Matching edges")
            event_ts, _ = match_chirp_edges(event_ts, ADC.global_timestamps[idx])

        # Update probe timestamps, starting state is not needed here
        PRB.update(event_ts - cont_ts[0], cont_ts[-1] - cont_ts[0])

        probe_times = PRB.global_timestamps[idx]
        ########## LOGGING #################################
        log_ts(event_ts, f"Event timestamps")
        log_ts(cont_ts, f"Continuous timestamps")
        log_ts(probe_times, f"Global segment")
        log_ts(np.concatenate(PRB.global_timestamps), f"Global")
        ########## LOGGING #################################

        adc_times = ADC.global_timestamps[idx]
        
        # Extract spikes based on continuous range
        cont_start, cont_end = PRB.intervals[idx]
        logger.info(f"Extracting spikes in interval: {cont_start:.5f} ... {cont_end:.5f} s")
        mask = (kilosort_spikes > cont_start) & (kilosort_spikes <= cont_end)
        # masks.append(mask) # DEBUG purpose
        
        probe_spikes = kilosort_spikes[mask]
        log_ts(probe_spikes, f"Extracted spikes")
        
        # Handle length mismatches
        min_length = min(len(probe_times), len(adc_times))
        if min_length < len(adc_times):
            logger.warning(f"  Truncating ADC timestamps. ADC timestamps: {len(adc_times)} -> {min_length}.")
            adc_times = adc_times[:min_length]
        elif min_length < len(probe_times):
            logger.warning(f"  Truncating Probe timestamps. Probe timestamps: {len(probe_times)} -> {min_length}.")
            PRB.global_timestamps[idx] = probe_times[:min_length]

        adc_global_timestamps.append(adc_times)

        # Interpolate/extrapolate to ADC time
        spl = make_interp_spline(x=probe_times, y=adc_times, k=1)
        adc_spikes = spl(probe_spikes)
        synced_spikes.append(adc_spikes)
        total_spikes_left -= adc_spikes.size
        log_ts(adc_spikes, "ADC interpolated spikes")
        logger.info(f"Synced spikes: {adc_spikes.size}/{kilosort_spikes.size}. Remaining spikes: {total_spikes_left}")
    
    probe_times = np.concatenate(PRB.global_timestamps)
    adc_times = np.concatenate(adc_global_timestamps)
    
    # np.save(save_dir / "adc_spikes.npy", np.concatenate(synced_spikes))
    # np.save(save_dir / "masks.npy", np.concatenate(masks))
    # np.save(save_dir / "intervals.npy", PRB.intervals)

    logger.info("="*60)

2025-11-03 18:21:59 | INFO     | Processing probe: ProbeA
2025-11-03 18:21:59 | INFO     | Extracting kilosort spikes
2025-11-03 18:22:00 | INFO     | ProbeA spikes: 0.0001 ... 33692.2234 s (110944227 samples)
2025-11-03 18:22:00 | INFO     | ============================================================
2025-11-03 18:22:00 | INFO     | Saving to: E:\kilosort_output\AA001_Day2\ProbeA
2025-11-03 18:22:00 | INFO     | Event timestamps: 12.0831 ... 3645.1092 s (7267 samples)
2025-11-03 18:22:00 | INFO     | Continuous timestamps: 11.6301 ... 3645.4102 s (109013404 samples)
2025-11-03 18:22:00 | INFO     | Global segment: 0.4530 ... 3633.4791 s (7267 samples)
2025-11-03 18:22:00 | INFO     | Global: 0.4530 ... 3633.4791 s (7267 samples)
2025-11-03 18:22:00 | INFO     | Extracting spikes in interval: 0.00000 ... 3633.78010 s
2025-11-03 18:22:00 | INFO     | Extracted spikes: 0.0001 ... 3633.7801 s (9688758 samples)
2025-11-03 18:22:00 | INFO     | ADC interpolated spikes: -0.0032 ... 3633.776

In [247]:
np.vstack((probe_times, adc_times)).T

array([[4.54051533e-01, 4.49691452e-01],
       [9.54034867e-01, 9.49707952e-01],
       [1.45405153e+00, 1.44969145e+00],
       ...,
       [3.42480978e+04, 3.42744301e+04],
       [3.42482431e+04, 3.42745737e+04],
       [3.42483853e+04, 3.42747177e+04]], shape=(175516, 2))

In [242]:
probe_filter = ['ProbeA', 'ProbeB']
probe_timestamps = {k:d for k,d in ps["timestamps"].items() if k != "OneBox-ADC" and k in probe_filter}

masks = {}
for probe, paths in probe_timestamps.items():
    PRB = Timestamps(name=probe, fs=30000.0, t_start=0.0)
    ADC_timestamps = []
    logger.info(f"Processing probe: {probe}")

    logger.info("Extracting kilosort spikes")
    kilosort_spikes = ks_spikes[probe] / PRB.fs
    log_ts(kilosort_spikes, f"{probe} spikes")
    logger.info('='*60)

    save_dir = Path(p['local_output'] / probe)
    save_dir.mkdir(parents=True, exist_ok=True)
    logger.info(f"Saving to: {save_dir}")

    # Compute global timestamps
    for idx, (ev_path, cont_path) in enumerate(zip(paths['event'], paths['cont'])):
        event_ts, cont_ts, states = load_events(ev_path, cont_path)

        # Handle state mismatches
        if states[0] != ADC.starting_states[idx]:
            logger.warning(f"State mismatch between {probe} and ADC")
            logger.info("Matching edges")
            event_ts, _ = match_chirp_edges(event_ts, ADC.global_timestamps[idx])

        # Update probe timestamps, starting state is not needed here
        PRB.update(event_ts - cont_ts[0], cont_ts[-1] - cont_ts[0])

        probe_times = PRB.global_timestamps[idx]
        ########## LOGGING #################################
        log_ts(event_ts, f"Event timestamps")
        log_ts(cont_ts, f"Continuous timestamps")
        log_ts(probe_times, f"Global segment")
        log_ts(np.concatenate(PRB.global_timestamps), f"Global")
        ########## LOGGING #################################

        adc_times = ADC.global_timestamps[idx]
        
        # Handle length mismatches
        min_length = min(len(probe_times), len(adc_times))
        if min_length < len(adc_times):
            logger.warning(f"  Truncating ADC timestamps. ADC timestamps: {len(adc_times)} -> {min_length}.")
            adc_times = adc_times[:min_length]
        elif min_length < len(probe_times):
            logger.warning(f"  Truncating Probe timestamps. Probe timestamps: {len(probe_times)} -> {min_length}.")
            PRB.global_timestamps[idx] = probe_times[:min_length]

        ADC_timestamps.append(adc_times)
    # Interpolate/extrapolate to ADC time
    probe_times = np.concatenate(PRB.global_timestamps)
    adc_times = np.concatenate(ADC_timestamps)
    spl = make_interp_spline(x=probe_times, y=adc_times, k=1)
    adc_spikes = spl(kilosort_spikes)
    log_ts(adc_spikes, "ADC interpolated spikes")
    logger.info(f"Synced spikes: {adc_spikes.size}/{kilosort_spikes.size}. Remaining spikes: {kilosort_spikes.size - adc_spikes.size}")
    np.save(save_dir / "adc_spikes2.npy", adc_spikes)
    # np.save(save_dir / "adc_spikes.npy", np.concatenate(synced_spikes))
    # np.save(save_dir / "masks.npy", np.concatenate(masks))
    # np.save(save_dir / "intervals.npy", PRB.intervals)

    logger.info("="*60)

2025-11-03 18:19:29 | INFO     | Processing probe: ProbeA
2025-11-03 18:19:29 | INFO     | Extracting kilosort spikes
2025-11-03 18:19:29 | INFO     | ProbeA spikes: 0.0001 ... 33692.2234 s (110944227 samples)
2025-11-03 18:19:29 | INFO     | ============================================================
2025-11-03 18:19:29 | INFO     | Saving to: E:\kilosort_output\AA001_Day2\ProbeA
2025-11-03 18:19:30 | INFO     | Event timestamps: 12.0831 ... 3645.1092 s (7267 samples)
2025-11-03 18:19:30 | INFO     | Continuous timestamps: 11.6301 ... 3645.4102 s (109013404 samples)
2025-11-03 18:19:30 | INFO     | Global segment: 0.4530 ... 3633.4791 s (7267 samples)
2025-11-03 18:19:30 | INFO     | Global: 0.4530 ... 3633.4791 s (7267 samples)
2025-11-03 18:19:30 | INFO     | Event timestamps: 123.0811 ... 1712.5706 s (8742 samples)
2025-11-03 18:19:30 | INFO     | Continuous timestamps: 123.0339 ... 1712.8201 s (47689465 samples)
2025-11-03 18:19:30 | INFO     | Global segment: 3633.8273 ... 5223.

In [31]:
import spikeinterface as si
rec = si.load("E:\\kilosort_output\\AA001_Day2\\ProbeA\\concat")

In [32]:
rec

BinaryFolderRecording: 384 channels - 30.0kHz - 1 segments - 1,010,766,713 samples 
                       33,692.22s (9.36 hours) - int16 dtype - 722.96 GiB